# Nuclei Reconstruction

Solution author: Asandei Stefan-Alexandru

This solution uses a patch-based UNet, with a ResNet34 backbone. Due to the dataset's large and diverse resolutions, a fixed size UNet (with a resize at inference) has a highly limited performance.

In [25]:
import os
import sys
import io
import base64
import random
import math
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import cv2
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
from torchvision import transforms as T
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [26]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

ROOT_PATH = Path.cwd()
TRAIN_DIR = ROOT_PATH / "train"
TEST_DIR = ROOT_PATH / "test"

# Configuration
PATCH_SIZE = 384
PATCH_STRIDE = 192  # 50% overlap
MIN_NUCLEI_RATIO = 0.02  # Min fraction of nuclei pixels in a patch
BATCH_SIZE = 8
NUM_EPOCHS = 60
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
VAL_SPLIT = 0.2

print(f"Patch size: {PATCH_SIZE}, Stride: {PATCH_STRIDE}")
print(f"Batch: {BATCH_SIZE}, Epochs: {NUM_EPOCHS}, LR: {LEARNING_RATE}")

Using device: cuda
Patch size: 384, Stride: 192
Batch: 8, Epochs: 60, LR: 0.0001


## Preprocessing — Stain Normalization & CLAHE

In [27]:
def macenko_normalize(img, alpha=1.0, beta=0.15):
    """Macenko stain normalization: maps image stain vectors to a reference standard."""
    ref_he = np.array([
        [0.5626, 0.7201, 0.4062],
        [0.2159, 0.8012, 0.5581],
    ])
    ref_he /= np.linalg.norm(ref_he, axis=1, keepdims=True)
    ref_max_c = np.array([1.9705, 1.0308])

    img = np.maximum(img.astype(np.float64), 1.0)
    od = -np.log(img / 255.0).reshape(-1, 3)

    mask = (od >= beta).any(axis=1)
    if mask.sum() < 10:
        return img.astype(np.uint8).reshape(img.shape)

    _, _, vh = np.linalg.svd(od[mask], full_matrices=False)
    v = vh[:2].T

    proj = od[mask] @ v
    phi = np.arctan2(proj[:, 1], proj[:, 0])
    phi_min, phi_max = np.percentile(phi, [alpha, 100 - alpha])

    stain_vectors = v @ np.array([
        [np.cos(phi_min), np.cos(phi_max)],
        [np.sin(phi_min), np.sin(phi_max)],
    ])
    stain_vectors = np.stack(sorted(stain_vectors.T, key=lambda vec: vec[2], reverse=True))

    concentrations = (np.linalg.pinv(stain_vectors.T) @ od.T).T
    concentrations = concentrations / (np.percentile(concentrations, 99, axis=0) + 1e-8)
    concentrations = concentrations * ref_max_c

    normalized_od = (ref_he.T @ concentrations.T).T
    normalized = np.exp(-normalized_od) * 255.0
    return np.clip(normalized, 0, 255).astype(np.uint8).reshape(img.shape)


def apply_clahe(img, clip_limit=2.0, tile_grid_size=(8, 8)):
    """Apply CLAHE to enhance local contrast on the luminance channel."""
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    l = clahe.apply(l)
    lab = cv2.merge((l, a, b))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def preprocess_image(image_rgb):
    """Full preprocessing pipeline: Macenko normalization + CLAHE."""
    img = macenko_normalize(image_rgb, alpha=1.0, beta=0.15)
    img = apply_clahe(img, clip_limit=2.0, tile_grid_size=(8, 8))
    return img

## Patch-Based Dataset

In [28]:
def extract_patches(image_rgb, mask, patch_size=PATCH_SIZE, stride=PATCH_STRIDE,
                    min_nuclei_ratio=MIN_NUCLEI_RATIO):
    """Extract overlapping patches, filtering patches with too few nuclei."""
    h, w = image_rgb.shape[:2]
    patches_img, patches_mask = [], []

    # Handle images smaller than patch_size
    if h < patch_size or w < patch_size:
        image_rgb = cv2.resize(image_rgb, (max(w, patch_size), max(h, patch_size)))
        mask = cv2.resize(mask, (max(w, patch_size), max(h, patch_size)),
                          interpolation=cv2.INTER_NEAREST)
        h, w = image_rgb.shape[:2]

    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            patch_img = image_rgb[y:y + patch_size, x:x + patch_size]
            patch_mask = mask[y:y + patch_size, x:x + patch_size]

            if mask is not None:
                nuclei_ratio = (patch_mask > 127).mean()
                if nuclei_ratio < min_nuclei_ratio:
                    continue

            patches_img.append(patch_img)
            patches_mask.append(patch_mask)

    # Handle right and bottom edges
    if w > patch_size:
        for y in range(0, h - patch_size + 1, stride):
            x = w - patch_size
            patch_img = image_rgb[y:y + patch_size, x:x + patch_size]
            patch_mask = mask[y:y + patch_size, x:x + patch_size]
            if mask is None or (patch_mask > 127).mean() >= min_nuclei_ratio:
                patches_img.append(patch_img)
                patches_mask.append(patch_mask)

    if h > patch_size:
        for x in range(0, w - patch_size + 1, stride):
            y = h - patch_size
            patch_img = image_rgb[y:y + patch_size, x:x + patch_size]
            patch_mask = mask[y:y + patch_size, x:x + patch_size]
            if mask is None or (patch_mask > 127).mean() >= min_nuclei_ratio:
                patches_img.append(patch_img)
                patches_mask.append(patch_mask)

    if not patches_img:
        return np.empty((0, patch_size, patch_size, 3), dtype=np.uint8), \
               np.empty((0, patch_size, patch_size), dtype=np.uint8)

    return np.stack(patches_img), np.stack(patches_mask)


class PatchDataset(Dataset):
    """Extracts patches from a given list of image files with caching."""
    def __init__(self, image_files, transform=None, desc="Loading patches"):
        self.image_files = sorted(image_files)
        self.transform = transform
        self.patches = []
        self.patch_masks = []

        print(f"{desc}: {len(self.image_files)} images...")
        for img_path in tqdm(self.image_files, desc=desc):
            mask_path = Path(str(img_path).replace('.jpg', '_mask.png'))

            image_rgb = cv2.imread(str(img_path))
            image_rgb = cv2.cvtColor(image_rgb, cv2.COLOR_BGR2RGB)
            image_rgb = preprocess_image(image_rgb)

            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            mask = (mask > 127).astype(np.uint8) * 255

            patches_img, patches_mask = extract_patches(image_rgb, mask)

            for p_img, p_mask in zip(patches_img, patches_mask):
                self.patches.append(p_img)
                self.patch_masks.append(p_mask)

        print(f"Extracted {len(self.patches)} patches from {len(self.image_files)} images")

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        image = self.patches[idx]
        mask = self.patch_masks[idx]

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]

        mask = (mask > 0.5).float()
        return image, mask.unsqueeze(0)

## Model

In [29]:
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c, mid_c=None):
        super().__init__()
        mid_c = mid_c or out_c
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, mid_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(mid_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_c, skip_c, out_c):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_c, in_c, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_c + skip_c, out_c)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=True)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class ResNetUNet(nn.Module):
    """U-Net with pretrained ResNet34 encoder. Output matches input resolution."""
    def __init__(self, pretrained=True):
        super().__init__()
        resnet = torchvision.models.resnet34(weights="IMAGENET1K_V1" if pretrained else None)

        # Encoder:   H -> H/2 -> H/4 -> H/8 -> H/16 -> H/32
        self.enc0 = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.enc1 = nn.Sequential(resnet.maxpool, resnet.layer1)   # 64
        self.enc2 = resnet.layer2  # 128
        self.enc3 = resnet.layer3  # 256
        self.enc4 = resnet.layer4  # 512

        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)

        # Decoder:   H/32 -> H/16 -> H/8 -> H/4 -> H/2 -> H
        self.dec4 = DecoderBlock(1024, 256, 512)
        self.dec3 = DecoderBlock(512, 128, 256)
        self.dec2 = DecoderBlock(256, 64, 128)
        self.dec1 = DecoderBlock(128, 64, 64)
        self.dec0 = DecoderBlock(64, 3, 32)          # H/2 -> H with RGB skip

        self.final = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        e0 = self.enc0(x)      #  64, H/2
        e1 = self.enc1(e0)     #  64, H/4
        e2 = self.enc2(e1)     # 128, H/8
        e3 = self.enc3(e2)     # 256, H/16
        e4 = self.enc4(e3)     # 512, H/32

        b = self.bottleneck(e4)

        d4 = self.dec4(b, e3)  # 512, H/16
        d3 = self.dec3(d4, e2) # 256, H/8
        d2 = self.dec2(d3, e1) # 128, H/4
        d1 = self.dec1(d2, e0) #  64, H/2
        d0 = self.dec0(d1, x)  #  32, H   (skip connection from raw RGB input)

        return self.final(d0)

In [30]:
# === Loss Functions ===

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, preds, targets):
        preds = torch.sigmoid(preds)
        intersection = (preds * targets).sum()
        union = preds.sum() + targets.sum()
        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice


class BCEWithDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, preds, targets):
        bce = self.bce(preds, targets)
        dice = self.dice(preds, targets)
        return self.bce_weight * bce + self.dice_weight * dice


# === Augmentations ===

# ToTensorV2 from albumentations 1.x keeps uint8, so we normalize after
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ElasticTransform(alpha=100, sigma=10, p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.RandomGamma(gamma_limit=(80, 120), p=0.3),
    A.GaussNoise(var_limit=(5.0, 25.0), p=0.3),
    A.Blur(blur_limit=3, p=0.2),
    A.Normalize(mean=(0.0, 0.0, 0.0), std=(1.0, 1.0, 1.0), max_pixel_value=255.0),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Normalize(mean=(0.0, 0.0, 0.0), std=(1.0, 1.0, 1.0), max_pixel_value=255.0),
    ToTensorV2(),
])

## Training

In [31]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for images, masks in tqdm(loader, desc="Training"):
        images, masks = images.to(device), masks.to(device)

        loss = criterion(model(images), masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def validate_epoch(model, loader, device, threshold=0.5):
    model.eval()
    iou_sum = 0.0
    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            preds = (torch.sigmoid(model(images)) > threshold).float()
            inter = (preds * masks).sum(dim=(1, 2, 3))
            union = ((preds + masks) > 0).float().sum(dim=(1, 2, 3))
            iou_sum += (inter / (union + 1e-6)).sum().item()
    return iou_sum / len(loader.dataset)

In [32]:
# === Dataset & DataLoaders (split at image level) ===

all_image_files = sorted([
    f for f in TRAIN_DIR.glob("*.jpg") if "_mask" not in f.name
])
random.Random(seed).shuffle(all_image_files)

val_size = int(VAL_SPLIT * len(all_image_files))
train_files = all_image_files[val_size:]
val_files = all_image_files[:val_size]

train_ds = PatchDataset(train_files, transform=train_transform, desc="Loading train patches")
val_ds = PatchDataset(val_files, transform=val_transform, desc="Loading val patches")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=True)

print(f"Train patches: {len(train_ds)}, Val patches: {len(val_ds)}")

# === Model, Loss, Optimizer ===

model = ResNetUNet(pretrained=True).to(device)
criterion = BCEWithDiceLoss(bce_weight=0.5, dice_weight=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                              weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)

# === Training Loop ===

best_val_iou = 0.0
best_epoch = 0

for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_iou = validate_epoch(model, val_loader, device, threshold=0.5)
    scheduler.step()

    if val_iou > best_val_iou:
        best_val_iou = val_iou
        best_epoch = epoch + 1
        torch.save(model.state_dict(), "best_model.pth")

    print(f"Epoch {epoch + 1:3d}/{NUM_EPOCHS} | Loss: {train_loss:.4f} | "
          f"Val IoU: {val_iou:.4f} | Best: {best_val_iou:.4f} (ep {best_epoch})")
    print(f"          LR: {scheduler.get_last_lr()[0]:.2e}")

print(f"\nTraining complete. Best Val IoU: {best_val_iou:.4f} at epoch {best_epoch}")

Loading train patches: 134 images...


Loading train patches: 100%|██████████| 134/134 [00:14<00:00,  9.38it/s]


Extracted 1271 patches from 134 images
Loading val patches: 33 images...


Loading val patches: 100%|██████████| 33/33 [00:02<00:00, 14.09it/s]


Extracted 201 patches from 33 images
Train patches: 1271, Val patches: 201


...

Epoch  58/60 | Loss: 0.1432 | Val IoU: 0.6166 | Best: 0.6238 (ep 49)
          LR: 1.27e-06


Training: 100%|██████████| 158/158 [00:34<00:00,  4.53it/s]


Epoch  59/60 | Loss: 0.1434 | Val IoU: 0.6148 | Best: 0.6238 (ep 49)
          LR: 1.07e-06


Training: 100%|██████████| 158/158 [00:34<00:00,  4.58it/s]


Epoch  60/60 | Loss: 0.1428 | Val IoU: 0.6158 | Best: 0.6238 (ep 49)
          LR: 1.00e-06

Training complete. Best Val IoU: 0.6238 at epoch 49


## Inference — Sliding Window & TTA

In [33]:
# Load best model
model.load_state_dict(torch.load("best_model.pth", weights_only=True))
model.eval()

GAUSS_WINDOW = None


def create_gaussian_window(patch_size, sigma_ratio=0.3):
    """Gaussian window for smooth patch blending (numpy)."""
    center = (patch_size - 1) / 2
    sigma = patch_size * sigma_ratio
    xs = (np.arange(patch_size) - center) / sigma
    ys = (np.arange(patch_size) - center) / sigma
    gauss_1d_x = np.exp(-0.5 * xs ** 2)
    gauss_1d_y = np.exp(-0.5 * ys ** 2)
    return np.outer(gauss_1d_y, gauss_1d_x).astype(np.float32)


def predict_sliding_window(model, image_rgb, patch_size=PATCH_SIZE,
                           stride=PATCH_STRIDE, batch_size=BATCH_SIZE,
                           tta=True, threshold=0.5):
    """Sliding window prediction with TTA and Gaussian blending."""
    global GAUSS_WINDOW
    if GAUSS_WINDOW is None or GAUSS_WINDOW.shape[0] != patch_size:
        GAUSS_WINDOW = create_gaussian_window(patch_size)

    orig_h, orig_w = image_rgb.shape[:2]
    h, w = orig_h, orig_w

    # Pad small images so they're at least patch_size
    pad_h = max(0, patch_size - h)
    pad_w = max(0, patch_size - w)
    if pad_h > 0 or pad_w > 0:
        image_rgb = cv2.copyMakeBorder(
            image_rgb, 0, pad_h, 0, pad_w,
            borderType=cv2.BORDER_REFLECT_101
        )
        h, w = image_rgb.shape[:2]

    stride = min(stride, patch_size // 2)

    # TTA transforms
    tta_transforms = [
        lambda x: x,
        lambda x: torch.flip(x, dims=[-1]),
        lambda x: torch.flip(x, dims=[-2]),
        lambda x: torch.flip(torch.flip(x, dims=[-1]), dims=[-2]),
        lambda x: torch.rot90(x, k=1, dims=[-1, -2]),
        lambda x: torch.rot90(x, k=2, dims=[-1, -2]),
        lambda x: torch.rot90(x, k=3, dims=[-1, -2]),
        lambda x: torch.flip(torch.rot90(x, k=1, dims=[-1, -2]), dims=[-1]),
    ]
    tta_inv = [
        lambda x: x,
        lambda x: torch.flip(x, dims=[-1]),
        lambda x: torch.flip(x, dims=[-2]),
        lambda x: torch.flip(torch.flip(x, dims=[-1]), dims=[-2]),
        lambda x: torch.rot90(x, k=-1, dims=[-1, -2]),
        lambda x: torch.rot90(x, k=-2, dims=[-1, -2]),
        lambda x: torch.rot90(x, k=-3, dims=[-1, -2]),
        lambda x: torch.rot90(torch.flip(x, dims=[-1]), k=-1, dims=[-1, -2]),
    ]

    pred_map = np.zeros((h, w), dtype=np.float32)
    weight_map = np.zeros((h, w), dtype=np.float32)

    # Collect all patch coordinates
    coords = []
    y_starts = list(range(0, h - patch_size + 1, stride))
    if not y_starts or y_starts[-1] != h - patch_size:
        y_starts.append(max(0, h - patch_size))
    x_starts = list(range(0, w - patch_size + 1, stride))
    if not x_starts or x_starts[-1] != w - patch_size:
        x_starts.append(max(0, w - patch_size))

    for y in y_starts:
        for x in x_starts:
            coords.append((y, x))

    # Process in batches
    for i in range(0, len(coords), batch_size):
        batch_coords = coords[i:i + batch_size]
        patches = np.stack([
            image_rgb[y:y + patch_size, x:x + patch_size].astype(np.float32) / 255.0
            for y, x in batch_coords
        ])
        batch = torch.from_numpy(patches).permute(0, 3, 1, 2).float().to(device)

        batch_preds = []
        transforms_pair = zip(tta_transforms, tta_inv) if tta else [(tta_transforms[0], tta_inv[0])]
        for tta_fn, inv_fn in transforms_pair:
            with torch.no_grad():
                pred = torch.sigmoid(model(tta_fn(batch)))
                pred = inv_fn(pred)
                batch_preds.append(pred)

        batch_pred = torch.stack(batch_preds).mean(dim=0)

        for j, (y, x) in enumerate(batch_coords):
            pred_map[y:y + patch_size, x:x + patch_size] += (
                batch_pred[j, 0].cpu().numpy() * GAUSS_WINDOW
            )
            weight_map[y:y + patch_size, x:x + patch_size] += GAUSS_WINDOW

    weight_map[weight_map == 0] = 1.0
    pred_map = pred_map / weight_map

    # Crop back to original dimensions
    pred_map = pred_map[:orig_h, :orig_w]
    return (pred_map > threshold).astype(np.uint8) * 255


def postprocess_mask(mask, min_size=50, morph_close_radius=2):
    """Remove small components and apply morphological closing."""
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (morph_close_radius * 2 + 1,) * 2
    )
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    for label_idx in range(1, num_labels):
        if stats[label_idx, cv2.CC_STAT_AREA] < min_size:
            mask[labels == label_idx] = 0

    return mask

In [34]:
print("Running inference on test set...")
results = []

test_files = sorted(Path(TEST_DIR).glob("*.jpg"))
for img_path in tqdm(test_files, desc="Predicting"):
    image_rgb = cv2.imread(str(img_path))
    image_rgb = cv2.cvtColor(image_rgb, cv2.COLOR_BGR2RGB)
    orig_h, orig_w = image_rgb.shape[:2]

    # Preprocess
    processed = preprocess_image(image_rgb)

    # Predict
    mask = predict_sliding_window(model, processed, tta=True, threshold=0.5)

    # Match original dimensions
    if mask.shape != (orig_h, orig_w):
        mask = cv2.resize(mask, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)

    # Post-process
    mask = postprocess_mask(mask, min_size=50, morph_close_radius=2)

    # Encode to base85 PNG
    pred_img = Image.fromarray(mask, mode='L')
    buf = io.BytesIO()
    pred_img.save(buf, format="PNG")
    b85 = base64.b85encode(buf.getvalue()).decode("ascii")

    results.append({
        "datapointID": img_path.stem,
        "subtaskID": "1",
        "answer": b85,
    })

# Save submission
submission_path = "solution_submission.csv"
pd.DataFrame(results).to_csv(submission_path, index=False)
print(f"Saved {len(results)} predictions to {submission_path}")
print("Done!")

Running inference on test set...


Predicting: 100%|██████████| 42/42 [00:37<00:00,  1.13it/s]

Saved 42 predictions to solution_submission.csv
Done!
